# 01 — Baseline ResNet18 (frozen backbone)

ResNet18 con backbone congelado entrenado con la **misma metodología en fases** y la **misma
suite de corrupciones** que el modelo DeMemte E5. Sirve como referencia _fair 1:1_:
mismo split de datos, mismas corrupciones, presupuesto de épocas equivalente.

**Tres fases**:
1. _warmup_ del clasificador en datos limpios (3 ep)
2. _corrupt_ — train con corrupciones aleatorias p=0.7 (6 ep)
3. _joint_ — refinamiento con lr reducido (10 ep)

Activa `RUN_TRAINING = True` para entrenar desde cero (1× con GPU). Si está en `False` (default),
carga el checkpoint guardado en `out/baseline_best.pt` y solo recomputa la evaluación.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

In [ ]:
import json
from dataclasses import asdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dememte.config import BaselineConfig, resolve_data_dir
from dememte.data import build_loaders, seed_everything
from dememte.models import ResNetBaseline
from dememte.training import train_baseline_phased
from dememte.evaluation import evaluate_baseline_suite, signal_curve_rows_baseline
from dememte.io import save_checkpoint, load_checkpoint, write_json, write_csv, ensure_dir

RUN_TRAINING = False  # flip to True to train from scratch

cfg = BaselineConfig(freeze_backbone=True)
cfg.data_dir = resolve_data_dir(cfg)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ensure_dir(ROOT / 'notebooks' / '01_baseline' / 'out')
CKPT = OUT / 'baseline_best.pt'
seed_everything(cfg.seed)
print(json.dumps(asdict(cfg), indent=2))

## Datos

In [ ]:
tr_loader, va_loader, te_loader, meta = build_loaders(
    data_dir=cfg.data_dir,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    val_ratio=cfg.val_ratio,
    split_seed=cfg.split_seed,
    protocol=cfg.benchmark_protocol,
)
print(meta)

## Modelo

In [ ]:
model = ResNetBaseline(num_classes=cfg.num_classes, freeze_backbone=True).to(device)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'trainable params: {n_trainable:,} / {n_total:,}')

## Entrenamiento (opcional)

Si `RUN_TRAINING=False`, salta esta celda y carga el checkpoint en la siguiente.

In [ ]:
if RUN_TRAINING:
    model, best_val = train_baseline_phased(model, tr_loader, va_loader, cfg, device)
    save_checkpoint(model, CKPT, extra={'best_val': best_val, 'config': asdict(cfg)})
    print(f'saved {CKPT} (best_val={best_val:.4f})')
else:
    if CKPT.exists():
        payload = load_checkpoint(model, CKPT, device=device, strict=True)
        print('loaded:', CKPT, '| best_val:', payload.get('best_val'))
    else:
        print('WARNING: no checkpoint at', CKPT, '— flip RUN_TRAINING to True to train.')

## Evaluación clean + corrupt (suite 4×3)

In [ ]:
metrics = evaluate_baseline_suite(model, te_loader, device=device, return_predictions=True)
clean_record = metrics.pop('clean_record')
corrupt_records = metrics.pop('corruption_records')

predictions = clean_record.pop('predictions', [])
for rows in corrupt_records.values():
    for rec in rows:
        predictions.extend(rec.pop('predictions', []))

summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float))}
summary.update({'protocol': meta['protocol'], 'split_seed': meta['split_seed']})
write_json(summary, OUT / 'metrics.json')
write_csv(predictions, OUT / 'predictions.csv')
write_csv(signal_curve_rows_baseline('ResNet18-frozen-baseline', clean_record, corrupt_records), OUT / 'corrupt_curves.csv')
print(json.dumps(summary, indent=2))

## Curva de robustez por severidad

In [ ]:
curves = pd.read_csv(OUT / 'corrupt_curves.csv')
fig, ax = plt.subplots(figsize=(8, 5))
for corr, sub in curves[curves['corruption'] != 'clean'].groupby('corruption'):
    ax.plot(sub['severity'], sub['acc'], marker='o', label=corr)
clean_acc = curves[curves['corruption'] == 'clean']['acc'].iloc[0]
ax.axhline(clean_acc, linestyle='--', color='k', alpha=0.5, label=f'clean ({clean_acc:.3f})')
ax.set_xlabel('Severity')
ax.set_ylabel('Accuracy')
ax.set_title('Baseline ResNet18 (frozen) — robustness curves')
ax.grid(alpha=0.3); ax.legend()
ensure_dir(OUT / 'plots')
fig.savefig(OUT / 'plots' / 'robustness_curves.png', dpi=120, bbox_inches='tight')
plt.show()